## Step 3  
Input: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/  
Output: s3://thesis--ec331-s3/melted-price-bids/  

In [2]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3

def get_memory_usage():
    """Return the current memory usage of the process in GB."""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / (1024 ** 3)
    return memory_gb

# Columns we want to melt for price bids
PRICEBAND_COLS = [f"PRICEBAND{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=50_000, output_folder="", file_label=""):
    """
    Melts the PRICEBAND columns in chunks and writes each chunk directly to S3.
    Filenames are of the form:
        {output_folder}/{file_label}_partXXXX.parquet

    Returns a tuple: (list_of_chunk_paths, total_rows_processed)
    """
    print(f"Starting streaming melt of DataFrame with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    num_rows = len(df)
    num_chunks = (num_rows + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    total_rows_processed = 0
    chunk_paths = []
    
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, num_rows)
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk: {get_memory_usage():.2f} GB")
        
        # Extract a copy of the current chunk
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not in PRICEBAND_COLS
        id_vars_cols = [col for col in chunk.columns if col not in PRICEBAND_COLS]
        
        # Melt the PRICEBAND columns
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[c for c in PRICEBAND_COLS if c in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDPRICE"
        )
        
        del chunk
        gc.collect()
        
        # Extract numeric band from e.g. "PRICEBAND3"
        chunk_melted["BIDBAND"] = (
            chunk_melted["BIDBAND"]
            .str.extract(r"PRICEBAND(\d+)")
            .astype(int)
        )
        
        # Optional type transformations
        if "BIDTYPE" in chunk_melted.columns:
            chunk_melted["BIDTYPE"] = chunk_melted["BIDTYPE"].astype(str)
        if "DUID" in chunk_melted.columns:
            chunk_melted["DUID"] = chunk_melted["DUID"].astype(str)
        if "SETTLEMENTDATE" in chunk_melted.columns:
            chunk_melted["SETTLEMENTDATE"] = pd.to_datetime(chunk_melted["SETTLEMENTDATE"])
        
        # Drop rows where BIDPRICE is null
        chunk_melted.dropna(subset=["BIDPRICE"], inplace=True)
        
        chunk_len = len(chunk_melted)
        total_rows_processed += chunk_len
        
        # Construct output filename
        chunk_output = f"{output_folder.rstrip('/')}/{file_label}_part{i+1:04d}.parquet"
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            chunk_paths.append(chunk_output)
            print(f"  ✓ Wrote chunk {i+1}/{num_chunks} with {chunk_len} rows to {chunk_output}")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1}: {str(e)}")
        
        del chunk_melted
        gc.collect()
        print(f"Memory usage after chunk {i+1}: {get_memory_usage():.2f} GB")
    
    total_time = time.time() - start_time
    print(f"\nAll chunks processed in {total_time:.2f} seconds.")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB\n")
    
    return chunk_paths, total_rows_processed

def process_single_file(input_file, base_output_prefix):
    """
    Process a single Parquet file from S3 for price bids:
      - Mirror its subfolder structure from the input.
      - Write the melted output chunks into the corresponding subfolder under the output prefix.
    """
    print(f"\nProcessing single file: {input_file}")
    
    bucket_name = input_file.split("/")[2]
    key = "/".join(input_file.split("/")[3:])  # e.g. "de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/xxx.parquet"
    
    file_name = os.path.basename(key)               # e.g. "xxx.parquet"
    file_name_no_ext = os.path.splitext(file_name)[0]  # e.g. "xxx"
    
    parent_path = os.path.dirname(key)  # e.g. "de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet"
    
    # Remove the fixed prefix so that only the subfolder structure remains.
    prefix_remove = "de-duped-price-bids/"
    relative_subpath = parent_path[len(prefix_remove):].lstrip("/")
    
    # Final output folder, preserving the subfolder structure.
    if relative_subpath:
        output_folder = f"{base_output_prefix.rstrip('/')}/{relative_subpath}/"
    else:
        output_folder = f"{base_output_prefix.rstrip('/')}/"
    
    print(f"Output subfolder: {output_folder}")
    print(f"File label: {file_name_no_ext}")
    
    start_time = time.time()
    df = wr.s3.read_parquet(path=input_file)
    print(f"Read file in {time.time() - start_time:.2f} seconds. Shape: {df.shape}")
    print(f"Memory usage after reading: {get_memory_usage():.2f} GB")
    
    # Melt in chunks and write the melted output to S3
    chunk_files, total_rows = melt_and_write_chunks(
        df=df,
        chunk_size=50_000,
        output_folder=output_folder,
        file_label=file_name_no_ext
    )
    
    del df
    gc.collect()
    
    print(f"File done. Created {len(chunk_files)} chunk(s), total {total_rows} rows processed.")
    return chunk_files, total_rows

if __name__ == "__main__":
    print("PROCESSING PRICE BIDS (MELT PRICE BANDS)")
    print("INPUT: s3://thesis--ec331-s3/de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/<parquet-files>")
    print("OUTPUT: s3://thesis--ec331-s3/melted-price-bids/<matching-subfolders>/<chunks>\n")
    
    input_bucket = "thesis--ec331-s3"
    input_prefix = "de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/"
    output_prefix = "s3://thesis--ec331-s3/melted-price-bids/"
    
    s3_client = boto3.client("s3")
    paginator = s3_client.get_paginator("list_objects_v2")
    page_iterator = paginator.paginate(Bucket=input_bucket, Prefix=input_prefix)
    
    parquet_files = []
    for page in page_iterator:
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith(".parquet"):
                    parquet_files.append(f"s3://{input_bucket}/{key}")
    
    print(f"\nFound {len(parquet_files)} .parquet file(s) under {input_prefix}.\n")
    
    total_files_processed = 0
    overall_rows = 0
    
    for file_s3_path in parquet_files:
        print("=" * 80)
        print(f"Starting to process: {file_s3_path}")
        try:
            melted_files, row_count = process_single_file(file_s3_path, base_output_prefix=output_prefix)
            print(f"Finished: {row_count} rows, {len(melted_files)} chunk file(s).\n")
            total_files_processed += 1
            overall_rows += row_count
        except Exception as ex:
            print(f"Error: {ex}")
        print("=" * 80)
    
    print(f"\nAll done. Processed {total_files_processed} file(s) total. Overall rows processed: {overall_rows}\n")

PROCESSING PRICE BIDS (MELT PRICE BANDS)
INPUT: s3://thesis--ec331-s3/de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/<parquet-files>
OUTPUT: s3://thesis--ec331-s3/melted-price-bids/<matching-subfolders>/<chunks>


Found 1 .parquet file(s) under de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/.

Starting to process: s3://thesis--ec331-s3/de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/deduped.parquet

Processing single file: s3://thesis--ec331-s3/de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/deduped.parquet
Output subfolder: s3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/
File label: deduped
Read file in 0.22 seconds. Shape: (435, 33)
Memory usage after reading: 0.19 GB
Starting streaming melt of DataFrame with shape: (435, 33)
Current memory usage: 0.19 GB
Processing in 1 chunks of size 50000
Processing chunk 1/1 (rows 0 to 434)
Memor

In [ ]:
price_bids_df.columns

In [ ]:
# To find the first (earliest) datetime
first_datetime = price_bids_df['SETTLEMENTDATE'].min()

# To find the last (latest) datetime
last_datetime = price_bids_df['SETTLEMENTDATE'].max()

# Print the results
print(f"First datetime: {first_datetime}")
print(f"Last datetime: {last_datetime}")

In [4]:
import pandas as pd
import awswrangler as wr

# Path to the test sample
test_sample_path = "s3://thesis--ec331-s3/melted-price-bids/test_melted_sample.parquet"

# Read the file
try:
    df = wr.s3.read_parquet(path=test_sample_path)
    
    # Display basic information
    print(f"Successfully read file: {test_sample_path}")
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.columns.tolist())
    
    # Display a sample of rows
    print("\nSample data:")
    display(df.head(10))
    
    # Check value distributions if sample is large enough
    if len(df) > 10:
        print("\nBIDBBAND distribution:")
        display(df['BIDBAND'].value_counts().sort_index())
        
        # Get basic statistics on BIDPRICE
        print("\nBIDPRICE statistics:")
        display(df['BIDPRICE'].describe())
except Exception as e:
    print(f"Error reading Parquet file: {e}")

Successfully read file: s3://thesis--ec331-s3/melted-price-bids/test_melted_sample.parquet
Shape: (10000, 27)

Columns:
['I', 'BIDS', 'BIDDAYOFFER', '1', 'DUID', 'BIDTYPE', 'SETTLEMENTDATE', 'OFFERDATE', 'VERSIONNO', 'PARTICIPANTID', 'DAILYENERGYCONSTRAINT', 'REBIDEXPLANATION', 'MINIMUMLOAD', 'T1', 'T2', 'T3', 'T4', 'NORMALSTATUS', 'LASTCHANGED', 'ENTRYTYPE', 'REBID_EVENT_TIME', 'REBID_AWARE_TIME', 'REBID_DECISION_TIME', 'REBID_CATEGORY', 'REFERENCE_ID', 'BIDBAND', 'BIDPRICE']

Sample data:


,I,BIDS,BIDDAYOFFER,1,DUID,BIDTYPE,SETTLEMENTDATE,OFFERDATE,VERSIONNO,PARTICIPANTID,...,NORMALSTATUS,LASTCHANGED,ENTRYTYPE,REBID_EVENT_TIME,REBID_AWARE_TIME,REBID_DECISION_TIME,REBID_CATEGORY,REFERENCE_ID,BIDBAND,BIDPRICE
0,D,BIDS,BIDDAYOFFER,1.0,ASNAES1,RAISE1SEC,2023/10/29 00:00:00,2023/10/27 14:31:21,1.0,AMALMASP,...,NaN,2023/10/27 14:31:21,DAILY,<NA>,<NA>,<NA>,<NA>,BE737C9E9E234F998B18FE4B2F9AC2E8,1,0.0
1,D,BIDS,BIDDAYOFFER,1.0,ASNAES1,RAISE1SEC,2023/10/31 00:00:00,2023/10/30 11:02:53,1.0,AMALMASP,...,NaN,2023/10/30 11:02:53,DAILY,<NA>,<NA>,<NA>,<NA>,a529df1f-c7ac-45c5-8995-4fae5b45ce46,1,0.0
2,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/08 00:00:00,2023/10/05 15:35:07,1.0,ENOCMASP,...,NaN,2023/10/05 15:35:07,DAILY,<NA>,<NA>,<NA>,<NA>,2370F99E2EF34763BC4001A791AD49B9,1,0.0
3,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/09 00:00:00,2023/10/06 14:19:35,1.0,ENOCMASP,...,NaN,2023/10/06 14:19:35,DAILY,15:15:00,<NA>,<NA>,<NA>,8f8661ba-0579-4ccb-9c99-80ba0f1f9a5d,1,0.0
4,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/09 00:00:00,2023/10/09 13:10:27,1.0,ENOCMASP,...,NaN,2023/10/09 13:10:27,REBID,13:08:00,<NA>,<NA>,<NA>,fb4c0f11-359e-47bf-a0ab-634a117b4390,1,0.0
5,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/10 00:00:00,2023/10/09 13:48:09,1.0,ENOCMASP,...,NaN,2023/10/09 13:48:09,REBID,13:45:00,<NA>,<NA>,<NA>,99c77c3c-d57e-4728-a8cc-af35e4dac82a,1,0.0
6,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/11 00:00:00,2023/10/10 08:34:44,1.0,ENOCMASP,...,NaN,2023/10/10 08:34:44,DAILY,08:30:00,<NA>,<NA>,<NA>,130b5977-aab4-4e62-b10d-66bc0d092b0c,1,0.5
7,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/12 00:00:00,2023/10/11 08:15:20,1.0,ENOCMASP,...,NaN,2023/10/11 08:15:20,DAILY,08:15:00,<NA>,<NA>,<NA>,feba9199-5ad5-4d39-bc75-5a23374e106d,1,0.5
8,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/12 00:00:00,2023/10/11 14:08:46,1.0,ENOCMASP,...,NaN,2023/10/11 14:08:46,REBID,14:05:00,<NA>,<NA>,<NA>,8e22789f-0ac2-462a-95c6-de2cbcf8c3dc,1,0.5
9,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/12 00:00:00,2023/10/12 09:49:17,1.0,ENOCMASP,...,NaN,2023/10/12 09:49:17,REBID,09:48:00,09:48:00,09:48:00,E,272b4135-b2ab-4f93-882a-07eab01f31da,1,0.5



BIDBBAND distribution:


BIDBAND
1     1000
2     1000
3     1000
4     1000
5     1000
6     1000
7     1000
8     1000
9     1000
10    1000
Name: count, dtype: Int64


BIDPRICE statistics:


count    10000.000000
mean      1603.498327
std       4779.386097
min          0.000000
25%          1.160000
50%          8.000000
75%         49.950000
max      16600.000000
Name: BIDPRICE, dtype: float64

Successfully read manifest file with 9 entries


,file_path,file_name,batch_number,part_number
0,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,1
1,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,2
2,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,3
3,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,4
4,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,5
5,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,6
6,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,7
7,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,8
8,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,9


In [ ]:
s3://thesis--ec331-s3/melted-price-bids/
